# 14. Build the Neural Network - deep dive

*Adapted from the official [Build the Neural Network](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html) tutorial. Complements `04_building_neural_networks.ipynb` - same `nn.Module` pattern as the blog, but this notebook breaks down each layer type individually (`nn.Flatten`, `nn.Linear`, `nn.ReLU`, `nn.Sequential`, `nn.Softmax`) on real image-shaped input, and shows the modern `torch.accelerator` device API.*

## Get a device for training

The unified `torch.accelerator` API (CUDA/MPS/XPU/MTIA in one call) is the modern replacement for checking each backend by hand - this is exactly what `utils/device.py` wraps for the rest of this primer:

In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA Available: True
GPU: Tesla T4


In [4]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Define the class

```mermaid
flowchart TD
    NN["NeuralNetwork (nn.Module)"] --> Flat["self.flatten<br/>(nn.Flatten)"]
    NN --> Stack["self.linear_relu_stack<br/>(nn.Sequential)"]
    Stack --> L1["[0] Linear(784, 512)"]
    Stack --> A1["[1] ReLU"]
    Stack --> L2["[2] Linear(512, 512)"]
    Stack --> A2["[3] ReLU"]
    Stack --> L3["[4] Linear(512, 10)"]
```

`nn.Module`s nest arbitrarily — `linear_relu_stack` is itself a module (`nn.Sequential`) living
inside the outer `NeuralNetwork` module. `named_parameters()` walks this whole tree and gives each
leaf parameter a dotted name reflecting its position (e.g.
`linear_relu_stack.0.weight`).

In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


model = NeuralNetwork().to(device)
print(model)


NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Never call `model.forward(x)` directly - always call `model(x)`, which runs some important bookkeeping around `forward` for you (hooks, etc.):

In [6]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")


Predicted class: tensor([4], device='cuda:0')


## Model layers, one at a time

Take a sample minibatch of 3 fake 28x28 images and trace it through each layer type.

In [7]:
input_image = torch.rand(3, 28, 28)
print(input_image.size())


torch.Size([3, 28, 28])


**`nn.Flatten`** collapses each 2D 28x28 image into a contiguous 784-length vector (keeping the batch dimension at dim 0):

In [8]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())   # torch.Size([3, 784])


torch.Size([3, 784])


**`nn.Linear`** applies a learned linear transformation (weights + bias):

In [9]:
layer1 = nn.Linear(in_features=28 * 28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())   # torch.Size([3, 20])


torch.Size([3, 20])


**`nn.ReLU`** (and other nonlinear activations) introduce nonlinearity between linear layers - without it, stacking linear layers would collapse into a single linear layer, unable to model complex relationships:

In [10]:
print(f"Before ReLU: {hidden1}\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")


Before ReLU: tensor([[-0.0226, -0.0883, -0.3180,  0.3006,  0.0523,  0.3384, -0.2495,  0.1132,
         -0.1701, -0.6442,  0.1701,  0.1609, -0.7269, -0.6420,  0.7303,  0.0654,
          0.1617, -0.1245,  0.2733,  0.0352],
        [-0.2819,  0.0419, -0.2165,  0.0705,  0.0224, -0.3063,  0.1140,  0.3966,
          0.1741, -0.4011,  0.4469,  0.2543, -0.8714, -0.5958,  0.3661,  0.1435,
          0.0521, -0.3012, -0.1515,  0.1902],
        [-0.3826, -0.1285, -0.5580,  0.1100, -0.3216,  0.0755,  0.2403,  0.3377,
         -0.2717, -0.6501,  0.2749, -0.1259, -0.4032, -0.3872,  0.3770,  0.2305,
         -0.3228, -0.0831,  0.3149,  0.2311]], grad_fn=<AddmmBackward0>)

After ReLU: tensor([[0.0000, 0.0000, 0.0000, 0.3006, 0.0523, 0.3384, 0.0000, 0.1132, 0.0000,
         0.0000, 0.1701, 0.1609, 0.0000, 0.0000, 0.7303, 0.0654, 0.1617, 0.0000,
         0.2733, 0.0352],
        [0.0000, 0.0419, 0.0000, 0.0705, 0.0224, 0.0000, 0.1140, 0.3966, 0.1741,
         0.0000, 0.4469, 0.2543, 0.0000, 0.0000, 0.366

**`nn.Sequential`** is an ordered container - data flows through each module in the order given, letting you assemble a quick network without writing a custom `forward`:

In [11]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10),
)
input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)
print(logits.shape)


torch.Size([3, 10])


**`nn.Softmax`** scales raw logits (`[-inf, inf]`) to `[0, 1]` values that sum to 1 along `dim` - interpretable class probabilities:

In [12]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)
print(pred_probab)
print(pred_probab.sum(dim=1))   # each row sums to ~1.0


tensor([[0.1049, 0.0884, 0.0883, 0.0939, 0.0818, 0.0959, 0.1110, 0.1230, 0.1186,
         0.0942],
        [0.1062, 0.0899, 0.0865, 0.0915, 0.0792, 0.1031, 0.1161, 0.1153, 0.1222,
         0.0899],
        [0.1084, 0.0917, 0.0889, 0.0949, 0.0706, 0.1102, 0.1013, 0.1103, 0.1226,
         0.1012]], grad_fn=<SoftmaxBackward0>)
tensor([1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


## Model parameters

Subclassing `nn.Module` automatically tracks every parameter you declare, accessible via `.parameters()` or `.named_parameters()`:

In [13]:
print(f"Model structure: {model}\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values: {param[:2]}\n")


Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)

Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values: tensor([[ 0.0078, -0.0341, -0.0090,  ...,  0.0223, -0.0205,  0.0089],
        [-0.0027, -0.0285, -0.0341,  ..., -0.0194, -0.0356, -0.0135]],
       device='cuda:0', grad_fn=<SliceBackward0>)

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values: tensor([0.0004, 0.0090], device='cuda:0', grad_fn=<SliceBackward0>)

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values: tensor([[ 0.0349, -0.0322, -0.0203,  ...,  0.0370, -0.0280,  0.0220],
        [ 0.0188, -0.0380, -0.0013,  ...,  0.0378, -0.0125, -0.0071]],
       device='cuda:0', grad_fn=<SliceBackw